# Applicazione di ELIta al corpus r/Italia — keyword *notizie*

Questo notebook applica il lessico ELIta (originale e versioni ricalcolate) ai commenti raccolti da r/Italia con keyword **notizie**.

## Import e configurazione

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from Fase3.support import (
    BASIC_EMOTIONS, SEVEN_EMOTIONS, POSITIVE, NEGATIVE, EMOTION_COLORS,
    load_corpus, load_all_matrices,
    detect_emotions, apply_corpus_mean_norm,
    plot_emotion_bars, plot_emotion_grid,
)

CORPUS_CSV   = Path('corpus_Italia_notizie.csv')
TOKENS_CSV   = Path('tokens_Italia_notizie.csv')
ELITA_CSV    = Path('../Fase1/ELIta_INTENSITY_Matrix.csv')
ALPHA_02_CSV = Path('../Fase2/output_csv/elita_recalculated_0_2.csv')
ALPHA_05_CSV = Path('../Fase2/output_csv/elita_recalculated_0_5.csv')
ALPHA_08_CSV = Path('../Fase2/output_csv/elita_recalculated_0_8.csv')
OUTPUT_DIR   = Path('output_confronto')
OUTPUT_DIR.mkdir(exist_ok=True)

print('Configurazione caricata.')


Configurazione caricata.


## Caricamento corpus, token e matrici ELIta

In [2]:
df_corpus, df_tokens = load_corpus(CORPUS_CSV, TOKENS_CSV)
print('Corpus:', len(df_corpus), 'documenti |', 'Token:', len(df_tokens))
print('  post:', (df_corpus['type'] == 'post').sum(), '| commenti:', (df_corpus['type'] == 'comment').sum())


Corpus: 2520 documenti | Token: 84867
  post: 200 | commenti: 2320


In [3]:
MATRICES = load_all_matrices(ELITA_CSV, ALPHA_02_CSV, ALPHA_05_CSV, ALPHA_08_CSV)
df_elita_orig = MATRICES['Originale (α=0)']
print('Matrici:', list(MATRICES.keys()))


Matrici: ['Originale (α=0)', 'Ibrido (α=0.2)', 'Ibrido (α=0.5)', 'Ibrido (α=0.8)']


## Funzione base e prima analisi (raw)

Per ogni commento sommiamo i vettori emotivi di tutti i lemmi ADJ+NOUN+VERB trovati in ELIta.
Nessun filtro, nessuna normalizzazione.

In [4]:
df_raw     = detect_emotions(df_corpus, df_tokens, df_elita_orig)
counts_raw = df_raw['dominant_emotion'].value_counts()
total      = len(df_raw)

df_raw.to_csv(OUTPUT_DIR / 'notizie_emotion_results_raw.csv', index=False)
print(f'Salvato: {OUTPUT_DIR / "notizie_emotion_results_raw.csv"}')

print('Distribuzione emozione dominante — raw:')
for e in BASIC_EMOTIONS + ['neutrale']:
    n = counts_raw.get(e, 0)
    print('{:<15s} {:>4d} ({:>4.1f}%) {}'.format(e, n, n/total*100, '█'*int(n/total*40)))


Salvato: output_confronto/notizie_emotion_results_raw.csv
Distribuzione emozione dominante — raw:
gioia            339 (13.5%) █████
tristezza         95 ( 3.8%) █
rabbia            94 ( 3.7%) █
paura            123 ( 4.9%) █
disgusto          17 ( 0.7%) 
fiducia          115 ( 4.6%) █
sorpresa          76 ( 3.0%) █
aspettativa     1467 (58.2%) ███████████████████████
neutrale         194 ( 7.7%) ███


In [5]:
plot_emotion_bars(counts_raw, total, 'Distribuzione emozione dominante — analisi raw').show()

**Osservazione**: aspettativa domina massicciamente (~60%).
Come documentato in ItEm (Pollacci 2015), alcune emozioni producono coseni più alti diventando "catalizzanti". Il primo passo è capire se questo bias è strutturale o semantico.
Seguiamo l'approccio di ItEm: ripetiamo l'analisi escludendo aspettativa.

## Passo 1 — Rimozione di aspettativa (corpus_7emo)

Come `corpus_sei_emo` in ItEm (che escludeva fiducia e attese), escludiamo aspettativa
per vedere la struttura emotiva sottostante. Non è il metodo finale — serve a esplorare.

In [6]:
df_7emo  = detect_emotions(df_corpus, df_tokens, df_elita_orig, emotions=SEVEN_EMOTIONS)
counts_7 = df_7emo['dominant_emotion'].value_counts()

plot_emotion_grid(
    panels=[
        ('Raw (8 emozioni)',              df_raw),
        ('Senza aspettativa (7 emozioni)',df_7emo),
    ],
    total=total,
    title='Raw vs corpus_7emo (senza aspettativa)',
    emotions=BASIC_EMOTIONS,
    height=500,
).show()


**Osservazione**: senza aspettativa emergono sorpresa e fiducia come dominanti.
Questa soluzione però è artificiosa: rimuove informazione invece di correggere il bias.
L'approccio corretto secondo ItEm è la normalizzazione.

## Passo 2 — Corpus_mean di ItEm (Formula 3.5)

In ItEm il corpus_mean normalizza **dopo** aver accumulato i punteggi grezzi di ogni
documento. La logica è:

```
Passo A  S_e(d)      = Σ_{w ∈ d} score(e, w)       ← score grezzo del doc (già il nostro "raw")
Passo B  μ_e         = (1/N) Σ_d S_e(d)             ← media corpus per emozione e
Passo C  S_e_norm(d) = S_e(d) / μ_e                  ← score normalizzato del documento
```

L'obiettivo: se un'emozione ha un valore medio alto su tutto il corpus (bias sistematico),
dividerla per la sua media la riporta su una scala comparabile con le altre emozioni.
Aspettativa ha μ ≈ 12.7, le altre emozioni hanno μ tra 4 e 10 — dopo la normalizzazione
aspettativa viene ridimensionata proporzionalmente di più.

In [7]:
df_corpus_mean, mu_raw = apply_corpus_mean_norm(df_raw)
counts_cm = df_corpus_mean['dominant_emotion'].value_counts()

print('Medie corpus per emozione (μ_e — score raw):')
for e in BASIC_EMOTIONS:
    print('  {:<15s}: {:.4f}'.format(e, mu_raw[e]))
print()
print('Distribuzione — corpus_mean ItEm (Formula 3.5):')
for e in BASIC_EMOTIONS + ['neutrale']:
    n  = counts_cm.get(e, 0)
    n0 = counts_raw.get(e, 0)
    print('{:<15s} {:>4d} ({:>4.1f}%)  [{:+d} vs raw]'.format(e, n, n/total*100, n-n0))


Medie corpus per emozione (μ_e — score raw):
  gioia          : 3.2605
  tristezza      : 2.1462
  rabbia         : 2.0582
  paura          : 2.3755
  disgusto       : 1.2822
  fiducia        : 3.0896
  sorpresa       : 2.5793
  aspettativa    : 4.1776

Distribuzione — corpus_mean ItEm (Formula 3.5):
gioia            397 (15.8%)  [+58 vs raw]
tristezza        250 ( 9.9%)  [+155 vs raw]
rabbia           177 ( 7.0%)  [+83 vs raw]
paura            211 ( 8.4%)  [+88 vs raw]
disgusto         461 (18.3%)  [+444 vs raw]
fiducia          308 (12.2%)  [+193 vs raw]
sorpresa         314 (12.5%)  [+238 vs raw]
aspettativa      208 ( 8.3%)  [-1259 vs raw]
neutrale         194 ( 7.7%)  [+0 vs raw]


In [8]:
results_raw_versions = {}
results_cm_ibridi    = {}
for vname, df_e in MATRICES.items():
    df_raw_v              = detect_emotions(df_corpus, df_tokens, df_e)
    results_raw_versions[vname] = df_raw_v
    results_cm_ibridi[vname], _ = apply_corpus_mean_norm(df_raw_v)

plot_emotion_grid(
    panels=[
        ('Raw (score originali)',            df_raw),
        ('corpus_mean ItEm (score orig)',    results_cm_ibridi['Originale (α=0)']),
        ('corpus_mean ItEm (ibrido α=0.5)', results_cm_ibridi['Ibrido (α=0.5)']),
        ('corpus_mean ItEm (ibrido α=0.8)', results_cm_ibridi['Ibrido (α=0.8)']),
    ],
    total=total,
    title='Corpus_mean ItEm (Formula 3.5): confronto versioni',
).show()

print('\nEffetto corpus_mean ItEm su aspettativa:')
print('{:<30s} | {:>8s} | {:>8s}'.format('Versione', 'Raw', 'corpus_mean'))
print('-' * 53)
for vname in MATRICES:
    c_raw = results_raw_versions[vname]['dominant_emotion'].value_counts()
    c_cm  = results_cm_ibridi[vname]['dominant_emotion'].value_counts()
    print('{:<30s} | {:>7.1f}% | {:>7.1f}%'.format(
        vname,
        c_raw.get('aspettativa', 0) / total * 100,
        c_cm.get('aspettativa',  0) / total * 100))



Effetto corpus_mean ItEm su aspettativa:
Versione                       |      Raw | corpus_mean
-----------------------------------------------------
Originale (α=0)                |    58.2% |     8.3%
Ibrido (α=0.2)                 |    59.4% |     8.7%
Ibrido (α=0.5)                 |    59.1% |     9.2%
Ibrido (α=0.8)                 |    55.1% |     9.7%


### Conclusione Passo 2 — corpus_mean ItEm su ELIta

Corpus_mean di ItEm (Formula 3.5) **riduce aspettativa** dal **60%** (raw) a **8%**.

**Meccanismo**: Dividendo per lo score raw, ogni emozione viene riportata su una scala comparabile. Le emozioni con valore medio alto nel corpus vengono penalizzate proporzionalmente alla loro diffusione.

**Questo è il metodo finale**: aggregazione grezza → corpus_mean (Formula 3.5).

## Diagnosi del bias — top driver di aspettativa

Per capire perché il raw mostra 60% di aspettativa, identifichiamo le parole che contribuiscono di più: **frequenza × score ELIta = contributo totale**.
Il corpus_mean normalizza questi contributi dividendo per lo score raw.

In [9]:
POS_FILTER = ['ADJ', 'NOUN', 'VERB']
df_filt   = df_tokens[df_tokens['pos'].isin(POS_FILTER)].copy()
elita_idx = set(df_elita_orig.index)

matched = sorted(set(df_filt['lemma']).intersection(elita_idx))
freq    = df_filt[df_filt['lemma'].isin(matched)]['lemma'].value_counts()

freq_df = freq.reset_index()
freq_df.columns = ['lemma', 'frequenza']
er = df_elita_orig.loc[matched, BASIC_EMOTIONS].reset_index()
er.columns = ['lemma'] + BASIC_EMOTIONS
freq_df = freq_df.merge(er, on='lemma', how='left')
freq_df['word_sum']    = freq_df[BASIC_EMOTIONS].sum(axis=1)
freq_df['contrib_asp'] = freq_df['frequenza'] * freq_df['aspettativa']

print('Top 25 parole per contributo ad ASPETTATIVA (frequenza × score):')
display(freq_df.nlargest(25, 'contrib_asp')[
    ['lemma', 'frequenza', 'aspettativa', 'word_sum', 'contrib_asp']
].round(3).reset_index(drop=True))

Top 25 parole per contributo ad ASPETTATIVA (frequenza × score):


,lemma,frequenza,aspettativa,word_sum,contrib_asp
0,fare,767,0.58,1.32,444.86
1,avere,491,0.58,2.86,284.78
2,notizia,184,0.71,3.12,130.64
3,vedere,234,0.54,2.63,126.36
4,dire,289,0.38,1.93,109.82
5,anno,202,0.54,1.79,109.08
6,pensare,141,0.75,3.75,105.75
7,dare,118,0.71,2.76,83.78
8,trovare,90,0.92,4.04,82.80
9,stare,123,0.54,2.03,66.42


**Osservazione**: le prime parole (*notizia*, *fare*, *avere*, *vedere*...) sono lemmi generici o legati al topic.
`fare` da sola vale **107 punti** (702 occorrenze × 0.58) — la keyword di ricerca è il principale driver di aspettativa nel raw.

Il corpus_mean ridimensiona questi contributi: dividendo per score raw, anche le parole con alta frequenza e score elevato vengono pesate proporzionalmente al loro ruolo medio nel corpus.

## Metodo finale: corpus_mean di ItEm su tutte le versioni ELIta

Applichiamo la **corpus_mean normalisation** (Formula 3.5 di ItEm) a tutte e quattro le versioni:

```
S_e_norm(d) = S_e(d) / μ_e     dove     μ_e = mean_d(S_e(d))
```

Nessun filtraggio lessicale — il bias di aspettativa è gestito interamente dalla normalizzazione.

In [10]:
results_final = {}
for vname, df_e in MATRICES.items():
    df_raw_v        = detect_emotions(df_corpus, df_tokens, df_e)
    results_final[vname], _ = apply_corpus_mean_norm(df_raw_v)
    matched = (df_raw_v['n_tokens_matched'] > 0).sum()
    print('{:<20s} | match: {:d}/{:d} ({:.0f}%)'.format(
          vname, matched, len(df_raw_v), matched/len(df_raw_v)*100))


Originale (α=0)      | match: 2326/2520 (92%)
Ibrido (α=0.2)       | match: 2322/2520 (92%)
Ibrido (α=0.5)       | match: 2322/2520 (92%)
Ibrido (α=0.8)       | match: 2322/2520 (92%)


In [11]:
plot_emotion_grid(
    panels=list(results_final.items()),
    total=total,
    title='Metodo finale: corpus_mean ItEm (Formula 3.5)',
).show()


## Confronto quantitativo: originale vs ricalcolato

Misuriamo l'impatto del ricalcolo su aspettativa e sulle altre emozioni, confrontando raw e corpus_mean.

In [12]:
plot_emotion_grid(
    panels=[
        ('Raw',                         df_raw),
        ('corpus_mean ItEm (metodo finale)', df_corpus_mean),
    ],
    total=total,
    title='Progressione: Raw → corpus_mean ItEm',
    height=500,
).show()


In [13]:
# Bilancio pos/neg
print('{:<35s} | {:>10s} | {:>10s}'.format('Configurazione','Positive','Negative'))
print('-'*62)
for label, df_r in [('Raw',              df_raw),
                    ('corpus_mean ItEm', df_corpus_mean)]:
    c = df_r['dominant_emotion'].value_counts()
    pos = sum(c.get(e, 0) for e in POSITIVE)
    neg = sum(c.get(e, 0) for e in NEGATIVE)
    print('{:<35s} | {:>9.1f}% | {:>9.1f}%'.format(label, pos/total*100, neg/total*100))

Configurazione                      |   Positive |   Negative
--------------------------------------------------------------
Raw                                 |      79.2% |      13.1%
corpus_mean ItEm                    |      48.7% |      43.6%


Possiamo vedere che la versione con α=0.5 riduce il gap tra aspettativa e le altre emozioni rispetto alla versione originale.

## 10. Tabella riassuntiva

In [14]:
# Tabella A: N. documenti e token con score > 0 per emozione
df_raw_orig = results_raw_versions['Originale (α=0)']
table_A = []
for e in BASIC_EMOTIONS:
    n_doc   = int((df_raw_orig[e] > 0).sum())
    doc_ids = set(df_raw_orig[df_raw_orig[e] > 0]['doc_id'])
    n_tok   = int(df_filt[
        df_filt['doc_id'].isin(doc_ids) &
        df_filt['lemma'].isin(set(df_elita_orig.index))
    ]['lemma'].count())
    table_A.append({'Emozione': e.capitalize(),
                    'N. Documenti (score>0)': n_doc,
                    'N. Token': n_tok})
print('Tabella A — N. documenti e token per emozione (metodo finale: corpus_mean):')
display(pd.DataFrame(table_A))

Tabella A — N. documenti e token per emozione (metodo finale: corpus_mean):


,Emozione,N. Documenti (score>0),N. Token
0,Gioia,2296,25004
1,Tristezza,2279,24983
2,Rabbia,2271,24965
3,Paura,2284,24994
4,Disgusto,2229,24898
5,Fiducia,2304,25020
6,Sorpresa,2311,25025
7,Aspettativa,2314,25028


In [15]:
dom = results_final['Originale (α=0)']['dominant_emotion'].value_counts()
tot = len(results_final['Originale (α=0)'])
table_B = [{'Emozione':e.capitalize(),
            'N. Documenti dom':int(dom.get(e,0)),
            '% totale':'{:.1f}%'.format(dom.get(e,0)/tot*100)}
           for e in BASIC_EMOTIONS+['neutrale']]
print('\nTabella B — Emozione dominante (corpus_mean ItEm, Formula 3.5):')
display(pd.DataFrame(table_B))


Tabella B — Emozione dominante (corpus_mean ItEm, Formula 3.5):


,Emozione,N. Documenti dom,% totale
0,Gioia,397,15.8%
1,Tristezza,250,9.9%
2,Rabbia,177,7.0%
3,Paura,211,8.4%
4,Disgusto,461,18.3%
5,Fiducia,308,12.2%
6,Sorpresa,314,12.5%
7,Aspettativa,208,8.3%
8,Neutrale,194,7.7%


## Conclusioni

### Percorso seguito

- **Raw**: aspettativa domina (~60%). Il bias è numerico: aspettativa ha il raw score, molto più alto rispetto alle altre emozioni.
- **Rimozione aspettativa** (corpus_7emo): utile per l'esplorazione, non come metodo finale.
- **Corpus_mean ItEm** (Formula 3.5): aspettativa scende a **8%**. Divide ogni score per la media di corpus di quell'emozione — le emozioni sistematicamente alte vengono penalizzate proporzionalmente.

### Effetto corpus_mean su aspettativa|

Il corpus_mean ridimensiona il bias numerico di aspettativa senza rimuovere alcuna parola dal lessico.
Il residuo (~8%) riflette la genuina caratterizzazione semantica del dominio *notizie*.